In [1]:
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship
from langchain_core.documents import Document
from typing import List, Dict, Any
from transformers import AutoTokenizer, AutoModel, DebertaV2Tokenizer, DebertaV2Model
import torch
from langchain_openai import ChatOpenAI


In [ ]:
def get_graph_doc_duplicate(path:str)->GraphDocument:
    with open(path, "rb") as file:
        import pickle
        graph_doc_raw = pickle.load(file)
    unique_nodes: List[Node] = []
    unique_relationships: List[Relationship] = []
    for gpd in graph_doc_raw:
        for node in gpd.nodes:
            if node not in unique_nodes:
                unique_nodes.append(node)
        for rel in gpd.relationships:
            if rel not in unique_relationships:
                unique_relationships.append(rel)
    
    graph_doc_dedup = GraphDocument(
        source=Document(page_content="合并去重后的图数据", metadata={}),
        nodes=unique_nodes,
        relationships=unique_relationships
    )
    return graph_doc_dedup

In [ ]:
data_path = "./data/"

# 读取原始图数据，进行去重处理，并保存去重后的图数据
graph_doc_dup = get_graph_doc_duplicate(data_path + "graph_doc_raw.pkl")
with open(data_path + "graph_doc_dedup.pkl", "wb") as file:
    import pickle
    pickle.dump(graph_doc_dup, file)





In [ ]:
import numpy as np
from embedding_deduplication import BailianEmbeddings
def get_cluster_graph_doc(graph_doc:GraphDocument, threshold:float=0.2)->dict:
    unique_nodes = graph_doc.nodes
    node_signatures = [f"({node.type}){node.id}" for node in unique_nodes]
    embedded = BailianEmbeddings(api_key="sk-aec12d19a21049a2a56fa32acb855dae")
    node_embeddings = embedded.embed_documents(node_signatures)
    node_embeddings = np.array(node_embeddings)

    from sklearn.metrics.pairwise import cosine_similarity
    similarity_matrix = cosine_similarity(node_embeddings) # shape: (n_nodes, n_nodes)
    distance_matrix = 1 - similarity_matrix
    distance_matrix = np.maximum(distance_matrix, 0)

    from sklearn.cluster import DBSCAN
    dbscan = DBSCAN(eps=threshold, min_samples=1, metric='precomputed') # 使用预计算的距离矩阵，进行聚类
    clusters = dbscan.fit_predict(distance_matrix) # shape: (n_nodes,), 值代表每个节点所属的簇标签

    print("节点聚类结果：", clusters)
    cluster_graph_doc = {}
    for idx, cluster_id in enumerate(clusters):
        if cluster_id not in cluster_graph_doc:
            cluster_graph_doc[cluster_id] = []
        cluster_graph_doc[cluster_id].append(unique_nodes[idx])

    return cluster_graph_doc

In [ ]:
# 用embedding进行初步分组
clustered_graph_doc = get_cluster_graph_doc(graph_doc_dup, threshold=0.25)
with open(data_path + "graph_doc_clustered.pkl", "wb") as file:
    import pickle
    pickle.dump(clustered_graph_doc, file)

for cluster_id, nodes in clustered_graph_doc.items():
    print(f"聚类 {cluster_id} 包含节点：")
    print([f"{node.type}:{node.id}" for node in nodes])
    print("\n")

In [ ]:
with open("./data/graph_doc_clustered.pkl", "rb") as file:
    import pickle
    clustered_graph_doc = pickle.load(file)
# 最多的聚类
max_cluster_id = max(clustered_graph_doc, key=lambda k: len(clustered_graph_doc[k]))
print(f"最大聚类ID: {max_cluster_id}, 节点数量: {len(clustered_graph_doc[max_cluster_id])}")

In [2]:
def generate_relationship_signature(rel: Relationship) -> str:
    properties = rel.properties or {}
    # properties_str = " ".join(f"{key}:{value}" for key, value in properties.items())
    # 只要properties中的condition字段
    properties_str = f"condition:{properties.get('relation_condition','')}"
    return f"{rel.source.type}  {rel.source.id} ({properties_str}) {rel.type}  {rel.target.type}  {rel.target.id}"
def generate_node_signature(node: Node) -> str:
    return f"{node.type} {node.id}"
def generate_node_signature_simple(node: Node) -> str:
    return f"{node.id}"

In [ ]:
# 从graph_doc得到node到relationship的映射
node_to_relations:Dict[str, List[Relationship]] = {}
with open("./data/graph_doc_dedup.pkl", "rb") as file:
    import pickle
    graph_doc_dedup = pickle.load(file)
for rel in graph_doc_dedup.relationships:
    source_node_key = generate_node_signature(rel.source)
    target_node_key = generate_node_signature(rel.target)
    if source_node_key not in node_to_relations:
        node_to_relations[source_node_key] = []
    if target_node_key not in node_to_relations:
        node_to_relations[target_node_key] = []
    node_to_relations[source_node_key].append(rel)
    node_to_relations[target_node_key].append(rel)

with open("./data/node_to_relations.pkl", "wb") as file:
    import pickle
    pickle.dump(node_to_relations, file)

In [ ]:
def find_positions(tokens, sub_tokens):
    for i in range(len(tokens) - len(sub_tokens) + 1):
        if tokens[i:i+len(sub_tokens)] == sub_tokens:
            # +1 因为模型输出第0位是 [CLS]
            return list(range(i+1, i+1+len(sub_tokens)))
    return []


def get_aggregated_word_vector(hidden_states, indices, num_layers_to_aggregate=4):
    """
    获取一个词在最后几层隐藏状态的聚合向量。
    """
    # 1. 选择最后num_layers_to_aggregate层的向量
    # hidden_states[-1]是最后一层, hidden_states[-4]是倒数第四层
    layers_to_use = hidden_states[-num_layers_to_aggregate:]

    # 2. 对每个选定的层，聚合目标词所有token的向量 (平均法)
    word_vectors_per_layer = []
    for layer_hidden_state in layers_to_use:
        # layer_hidden_state shape: [batch_size, seq_len, hidden_size]
        # 获取目标词所有token的向量
        token_vectors = layer_hidden_state[0, indices, :] # shape: [num_tokens, hidden_size]
        # 对这些token的向量求平均，得到该层中该词的单一表示(类似于池化)
        mean_vector = torch.mean(token_vectors, dim=0) # shape: [hidden_size]
        word_vectors_per_layer.append(mean_vector) #    shape: [hidden_size]

    # 3. 聚合来自不同层的词向量 (拼接法或求和/平均法)
    # 拼接法能保留更多信息，但维度会变高
    # final_vector = torch.cat(word_vectors_per_layer, dim=0)
    # 求和/平均法更简单，维度不变
    # final_vector = torch.stack(word_vectors_per_layer, dim=0).sum(dim=0)
    final_vector = torch.mean(torch.stack(word_vectors_per_layer, dim=0), dim=0) # shape: [hidden_size]
    
    return final_vector

def compute_word_similarity(model:AutoModel, tokenizer:AutoTokenizer, aim_word:str, src_sent:str, rep_sent:str, k:int=4):
    """
    Arguments:
        model: 预训练语言模型
        tokenizer: 预训练语言模型对应的tokenizer
        aim_word: 替换句子中的词
        src_sent: 包含aim_word的源句子
        rep_sent: 包含aim_word的替换句子
        k: 聚合最后k层的隐藏状态
    Returns:
        cos_similarity: 基于最后k层隐藏状态的余弦相似度
        emb_cos_similarity: 基于embedding层的余弦相似度
    """
    # 对源句子和替换句子进行tokenize
    src_input = tokenizer(src_sent, return_tensors="pt")
    rep_input = tokenizer(rep_sent, return_tensors="pt")

    # 获取源词和替换词的token ids
    src_sent_tokens = tokenizer.tokenize(src_sent)
    rep_sent_tokens = tokenizer.tokenize(rep_sent)
    aim_word_tokens = tokenizer.tokenize(aim_word)
    print("源句子tokens:", src_sent_tokens)
    print("替换句子tokens:", rep_sent_tokens)
    print("目标词tokens:", aim_word_tokens)
    # src_word_tokens = tokenizer.tokenize(src_word)
    # rep_word_tokens = tokenizer.tokenize(rep_word)

    pos_src = find_positions(src_sent_tokens, aim_word_tokens)
    pos_rep = find_positions(rep_sent_tokens, aim_word_tokens)
    print("源句子中目标词位置:", pos_src)
    print("替换句子中目标词位置:", pos_rep)

    # 获取模型输出的隐藏状态
    with torch.no_grad():
        src_outputs = model(**src_input)
        rep_outputs = model(**rep_input)

    # 聚合最后k层的词向量并计算余弦相似度
    src_last_k_layers = get_aggregated_word_vector(src_outputs.hidden_states, pos_src, num_layers_to_aggregate=k)
    rep_last_k_layers = get_aggregated_word_vector(rep_outputs.hidden_states, pos_rep, num_layers_to_aggregate=k)
    cos_similarity = torch.nn.functional.cosine_similarity(src_last_k_layers, rep_last_k_layers, dim=0) # 基于最后k层隐藏状态的余弦相似度

    src_embedding = src_outputs.hidden_states[0]
    rep_embedding = rep_outputs.hidden_states[0]
    src_emb_vec = src_embedding[0, pos_src, :]
    rep_emb_vec = rep_embedding[0, pos_rep, :]
    mean_src_emb_vec = torch.mean(src_emb_vec, dim=0)
    mean_rep_emb_vec = torch.mean(rep_emb_vec, dim=0)
    emb_cos_similarity = torch.nn.functional.cosine_similarity(mean_src_emb_vec, mean_rep_emb_vec, dim=0) # 基于embedding层的余弦相似度
    # print("embedding余弦相似度:", emb_cos_similarity.item())

    return cos_similarity.item(), emb_cos_similarity.item()

In [ ]:
# 利用bert模型进行判断聚类内的节点是否需要合并
def similiarity_degree(node1:Node, node2:Node, node_to_relations:Dict[str, List[Relationship]], model:AutoModel, tokenizer:AutoTokenizer, k:int=8):
    node1_str = generate_node_signature(node1)
    node2_str = generate_node_signature(node2)

    rels1 = node_to_relations.get(node1_str, [])
    rels2 = node_to_relations.get(node2_str, [])

    node1_str_simple = generate_node_signature_simple(node1)
    node2_str_simple = generate_node_signature_simple(node2)

    cos_similarities = []

    # embedding相似度只需要计算一次
    embedding_similarities = []

    for rel1 in rels1:
        for rel2 in rels2:
            if rel1 == rel2:
                continue
            rel1_str = generate_relationship_signature(rel1)
            rel2_str = generate_relationship_signature(rel2)

            src_word = node2_str_simple
            aim_word = node1_str_simple
            src_sent = rel1_str # node1的源关系
            rep_sent = rel2_str.replace(src_word, aim_word) # 将node2的源关系中的node2替换为node1
            print("比较关系：", src_sent,"  ", rep_sent)
            cos_similarity, embedding_similarity = compute_word_similarity(src_sent=src_sent, rep_sent=rep_sent, aim_word=aim_word, model=model, tokenizer=tokenizer, k=k)
            print(f"关系相似度: {cos_similarity}, embedding相似度: {embedding_similarity}")
            cos_similarities.append(cos_similarity)
            if embedding_similarities==[]:  # 只计算一次
                embedding_similarities.append(embedding_similarity)
    if cos_similarities:
        avg_cos_similarity = sum(cos_similarities) / len(cos_similarities)
        avg_embedding_similarity = sum(embedding_similarities) / len(embedding_similarities)
    else:
        avg_cos_similarity = 0.0
        avg_embedding_similarity = 0.0
    return avg_cos_similarity, avg_embedding_similarity


In [ ]:
model_dir = "/mnt/e/project/DataGraphX_Learn - 副本/bert/deberta-v3-base"

tokenizer = DebertaV2Tokenizer.from_pretrained(model_dir)
model = DebertaV2Model.from_pretrained(pretrained_model_name_or_path=model_dir, output_hidden_states=True)

In [ ]:
node1 = clustered_graph_doc[2][0]
node2 = clustered_graph_doc[2][1]
print(node1.type, node1.id)
print(node2.type, node2.id)
with open("./data/node_to_relations.pkl", "rb") as file:
    import pickle
    node_to_relations = pickle.load(file)
cos_similarity, embedding_similarity = similiarity_degree(node1, node2, node_to_relations, model, tokenizer, k=4)
print(f"节点相似度: {cos_similarity}, embedding相似度: {embedding_similarity}")


In [ ]:
# 对同一类节点进行计算并记录结果
with open("./data/graph_doc_clustered.pkl", "rb") as file:
    import pickle
    clustered_graph_doc = pickle.load(file)
with open("./data/node_to_relations.pkl", "rb") as file:
    import pickle
    node_to_relations = pickle.load(file)
results = []

# 在循环中增加一个进度条：
from tqdm import tqdm
for cluster_id, nodes in tqdm(clustered_graph_doc.items(), desc="处理聚类"):
    print(f"处理聚类{cluster_id}，节点数量: {len(nodes)}")
    if len(nodes) < 2:
        continue
    # results[cluster_id] = {}
    for i in range(len(nodes)):
        for j in range(i+1, len(nodes)):
            node1 = nodes[i]
            node2 = nodes[j]
            cos_similarity, embedding_similarity = similiarity_degree(node1, node2, node_to_relations, model, tokenizer, k=8)
            print(f"节点{node1.type}:{node1.id} 与 节点{node2.type}:{node2.id} 相似度: {cos_similarity}, embedding相似度: {embedding_similarity}")
            results.append({
                "node1": generate_node_signature_simple(node1),
                "node2": generate_node_signature_simple(node2),
                "cos_similarity": cos_similarity,
                "embedding_similarity": embedding_similarity
            })
            # results[cluster_id][(f"{node1.type}:{node1.id}", f"{node2.type}:{node2.id}")] = {
            #     "cos_similarity": cos_similarity,
            #     "embedding_similarity": embedding_similarity
            # }

"""
results结构：
[
    {
        "node1": str,
        "node2": str,
        "cos_similarity": float,
        "embedding_similarity": float
    },
    ...
]
"""
with open("./data/cluster_node_similarity_results(no_NodeType).pkl", "wb") as file:
    import pickle
    pickle.dump(results, file)

In [ ]:
with open("./data/cluster_node_similarity_results.pkl", "rb") as file:
    import pickle
    similarity_results = pickle.load(file)
# 查看结果信息
print("相似度结果数量:", len(similarity_results))
for res in similarity_results[:5]:
    print(res)
# 查看其中相似度最低的几个结果
similarity_results_sorted = sorted(similarity_results, key=lambda x: x["cos_similarity"])
print("相似度最低的几个结果:")
for res in similarity_results_sorted[:5]:
    print(res)


In [5]:
with open("./data/cluster_node_similarity_results.pkl", "rb") as file:
    import pickle
    similarity_results = pickle.load(file)
# 查看总数量
print("相似度结果总数量:", len(similarity_results))

with open("./data/llm_node_merge_decisions(qwen-plus).pkl", "rb") as file:
    import pickle
    merge_decisions = pickle.load(file)
# 查看数量
print("LLM合并决策结果总数量:", len(merge_decisions))
# 查看这两个中的数据是否一一对应
mismatch_count = 0
for sim_res, merge_dec in zip(similarity_results, merge_decisions):
    if (sim_res["node1"] != merge_dec["node1"]) or (sim_res["node2"] != merge_dec["node2"]):
        mismatch_count += 1
        print("不匹配的项:")
        print("相似度结果:", sim_res)
        print("合并决策:", merge_dec)
print("不匹配的项总数量:", mismatch_count)

相似度结果总数量: 461
LLM合并决策结果总数量: 461
不匹配的项总数量: 0


In [5]:
# 利用llm以问询的方式进行判断节点是否需要合并
from LLMNodeMerger import LLMNodeMerger, MergeDecision

# model:
# qwen-plus-2025-09-11
# qwen-max
# qwen3-max
# deepseek-v3.2
model_name = "deepseek-v3.2"
llm_qwen = ChatOpenAI(
        model=model_name,
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
        api_key="sk-aec12d19a21049a2a56fa32acb855dae",
    )
nodemerger = LLMNodeMerger(llm=llm_qwen,
                           function_calling=False,
                           pydantic_object=MergeDecision
                           )
"""
    info:[
        {
            "node1": Node_str,
            "node2": Node_str,
            "relations1": List[Relationship_str],
            "relations2": List[Relationship_str],
        },
        ...
    ]
"""
info = []
with open("./data/graph_doc_clustered.pkl", "rb") as file:
    import pickle
    clustered_graph_doc = pickle.load(file)
with open("./data/node_to_relations.pkl", "rb") as file:
    import pickle
    node_to_relations = pickle.load(file)

# 先计算前100个节点对
# count = 0
for cluster_id, nodes in clustered_graph_doc.items():
    if len(nodes) < 2:
        continue
    for i in range(len(nodes)):
        for j in range(i+1, len(nodes)):
            # count += 1
            # if count > 100:
            #     break
            node1 = nodes[i]
            node2 = nodes[j]
            node1_str = generate_node_signature(node1)
            node2_str = generate_node_signature(node2)
            rels1 = node_to_relations.get(node1_str, [])
            rels2 = node_to_relations.get(node2_str, [])
            rels1_str = [generate_relationship_signature(rel) for rel in rels1]
            rels2_str = [generate_relationship_signature(rel) for rel in rels2]
            info.append({
                "node1": node1_str,
                "node2": node2_str,
                "relations1": rels1_str,
                "relations2": rels2_str,
            })
merge_decisions = nodemerger.merge_nodes(info, 20)
with open(f"./data/llm_node_merge_decisions({model_name}).pkl", "wb") as file:
    import pickle
    pickle.dump(merge_decisions, file)

LLM原始输出: {'results': [{'node1': 'ComputerSystemComponent Kernel', 'node2': 'KernelArchitecture Monolithic kernel', 'can_merge': False}, {'node1': 'ComputerSystemComponent Kernel', 'node2': 'KernelArchitecture Microkernel', 'can_merge': False}, {'node1': 'ComputerSystemComponent Kernel', 'node2': 'KernelArchitecture Linux kernel', 'can_merge': False}, {'node1': 'ComputerSystemComponent Kernel', 'node2': 'Component Kernel', 'can_merge': True}, {'node1': 'ComputerSystemComponent Kernel', 'node2': 'Component kernel', 'can_merge': True}, {'node1': 'ComputerSystemComponent Kernel', 'node2': 'Component running kernel', 'can_merge': True}, {'node1': 'ComputerSystemComponent Kernel', 'node2': 'Entity Kernel directly', 'can_merge': False}, {'node1': 'ComputerSystemComponent Kernel', 'node2': 'ComputerSystemComponent Microkernel', 'can_merge': False}, {'node1': 'ComputerSystemComponent Kernel', 'node2': 'ComputerSystemComponent Monolithic kernel', 'can_merge': False}, {'node1': 'ComputerSystemCom

In [ ]:
import pandas as pd
model_names = ["qwen-plus", "qwen-max", "qwen3-max", "deepseek-v3.2"]

with open("./data/cluster_node_similarity_results(no_NodeType).pkl", "rb") as file:
    import pickle
    similarity_results = pickle.load(file)

for model_name in model_names:
    with open(f"./data/llm_node_merge_decisions({model_name}).pkl", "rb") as file:
        import pickle
        merge_decisions = pickle.load(file)
    
    for sim_res, merge_dec in zip(similarity_results, merge_decisions):
        # if (sim_res["node1"] != merge_dec["node1"]) or (sim_res["node2"] != merge_dec["node2"]):
        #     print("不匹配的项:")
        #     print("相似度结果:", sim_res)
        #     print("合并决策:", merge_dec)
        #     continue
        # 直接修改similarity_results，增加一个字段
        sim_res[f"merge_decision_{model_name}"] = merge_dec["can_merge"]

# 保存为.xlsx文件
df = pd.DataFrame(similarity_results)
df.to_excel("./data/node_merge_decision_comparison.xlsx", index=False)
        
        
    

KeyError: 'merge_decision'

In [7]:
import pandas as pd

model_name = "qwen-plus"
with open(f"./data/llm_node_merge_decisions({model_name}).pkl", "rb") as file:
    import pickle
    merge_decisions = pickle.load(file)

df = pd.DataFrame(merge_decisions)
output_path = f"./data/llm_node_merge_decisions({model_name}).xlsx"
df.to_excel(output_path, index=False)
